[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Deep-unlearning/smol-audio/blob/master/cohere_transcribe_streaming/Distill_Streaming_Student.ipynb)

# Distill a streaming student, in miniature

How do you turn an **offline** speech model (it needs the whole recording first) into a **streaming** one (it writes
as you speak)? This notebook walks through the recipe behind
[Cohere Streaming](https://huggingface.co/Steveeeeeeen/cohere-streaming), at toy scale, running the **real** code of
the repo: `distill_dataset.py`, `train.py` and `model.py`.

**The recipe in one paragraph.** A frozen **teacher** (the offline Cohere Transcribe model) transcribes lots of audio.
A **student** with a compatible Conformer structure and a *streaming* encoder (limited attention, causal convolutions,
a cache) learns to reproduce the teacher's transcript and the teacher's internal features from past audio and the current chunk, including its limited lookahead. The student starts from the teacher's weights, so training only has to *adapt* it.

**Steps**

1. **Relabel**: run `distill_dataset.py` so the teacher writes the training targets for a tiny slice of audio.
2. **Train**: build the student and train it for a few steps with a `transformers` `Trainer`
   (`DistillTrainer` / `DistillTrainingArguments` from `train.py`).
3. **Evaluate**: measure word error rate (WER) before and after.
4. **Stream**: load the checkpoint in `model.CohereStreaming` and caption a clip live.

> **This is a toy.** We train for a handful of steps on a handful of clips. It shows every moving part and runs in
> minutes, but it will **not** produce a good model. A useful model needs much more teacher-labelled audio and training.

**Hardware.** Use a GPU runtime. Step 1 can use a T4 with fp16 and a small teacher batch. Step 2 keeps a teacher,
student, gradients and optimizer state in memory for a 2B-parameter model: plan for an **A100 80 GB or H100**.
An A100 40 GB may work with the default `BATCH_SIZE=1`, depending on clip lengths. The training cells require a GPU
with native bf16 support. A CPU needs substantial RAM and is too slow for this walkthrough.

This example is part of [smol-audio](https://github.com/Deep-unlearning/smol-audio/tree/master/cohere_transcribe_streaming), adapted from [the upstream notebook contribution](https://github.com/Deep-unlearning/cohere-transcribe-streaming/tree/1e1b41ca259b85d0e2b257b6a794d1b9c1e1ef2d) ([PR #3](https://github.com/Deep-unlearning/cohere-transcribe-streaming/pull/3)).

## 0. Setup

The training code lives in the `cohere_transcribe_streaming` folder of [smol-audio](https://github.com/Deep-unlearning/smol-audio).
The cell below clones the repo in a fresh Colab runtime, or reuses your local checkout, then moves into that folder
and installs the dependencies from `requirements.txt`.


In [ ]:
from pathlib import Path
import subprocess

# Reuse a local checkout, or clone smol-audio in a fresh Colab runtime. Safe to rerun this cell.
here = Path.cwd()
if here.name == "cohere_transcribe_streaming" and (here / "train.py").is_file():
    example_dir = here
elif (here / "cohere_transcribe_streaming" / "train.py").is_file():
    example_dir = here / "cohere_transcribe_streaming"
else:
    checkout = here / "smol-audio"
    if not checkout.exists():
        subprocess.run(["git", "clone", "--quiet", "https://github.com/Deep-unlearning/smol-audio", str(checkout)], check=True)
    example_dir = checkout / "cohere_transcribe_streaming"
%cd {example_dir}
%pip install -r requirements.txt


Before downloading the model, sign in on the [Cohere Transcribe model page](https://huggingface.co/CohereLabs/cohere-transcribe-03-2026)
and accept its access conditions. Use a [Hugging Face read token](https://huggingface.co/settings/tokens) from that same account.

The cell below opens the login form only if no token is already available. In Colab, you can instead add an `HF_TOKEN`
secret in the Secrets panel and enable notebook access; the Hub library will use it automatically.


In [ ]:
from huggingface_hub import get_token, notebook_login

if get_token() is None:
    notebook_login()


Now the imports, and a check of which device we have.

In [ ]:
import gc
import json
import sys
import time
from pathlib import Path

import numpy as np
import torch

REPO = Path.cwd()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

device = "cuda" if torch.cuda.is_available() else "cpu"
NATIVE_BF16 = device == "cuda" and torch.cuda.is_bf16_supported(including_emulation=False)
if NATIVE_BF16:
    TEACHER_DTYPE = "bfloat16"
elif device == "cuda":
    TEACHER_DTYPE = "float16"
else:
    TEACHER_DTYPE = "float32"

print("repo:", REPO)
print("device:", device)
print("teacher inference dtype:", TEACHER_DTYPE)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")

Knobs for this walkthrough. Keep them small: the real run uses thousands of hours of audio and thousands of steps.

In [ ]:
N_TRAIN_CLIPS = 16
N_HELD_OUT = 8
MAX_STEPS = 20
BATCH_SIZE = 1
LOOKAHEADS = "0,1,3,6,13"
SRC_DATASET = "hf-internal-testing/librispeech_asr_dummy"
SRC_CONFIG = "clean"
SRC_SPLIT = "validation"

WORK = REPO / "work"
RELABEL_DIR = WORK / "relabel"
CKPT = WORK / "tiny-student"
WORK.mkdir(exist_ok=True)

## 1. Teacher relabelling with `distill_dataset.py`

**The idea.** The student must imitate the *teacher*, so the best training target for a clip is what the **teacher**
says about it, not the human reference. The targets are then exactly in the teacher's style (casing, punctuation,
number formats), and the student never has to learn anything the teacher cannot do. This is called *sequence-level
distillation*.

`distill_dataset.py` does this in one command. It reads a Hub dataset, decodes the audio, asks the teacher for the
transcript of every clip (as **token ids**, so nothing is re-tokenised later) and writes an ordinary Hugging Face
`datasets` dataset with one row per clip:

| column | meaning |
|---|---|
| `audio` | the clip as 16 kHz FLAC (so the dataset is complete training data on its own) |
| `language`, `duration` | language code given to the teacher, length in seconds |
| `teacher_ids`, `teacher_text` | the teacher's tokens and the same tokens as text |
| `reference` | the source dataset's own transcript, if any (handy to filter bad rows) |

Here we label the first `N_TRAIN_CLIPS` clips of a tiny LibriSpeech sample. This downloads and runs the 2B teacher once
(a few minutes the first time). For the real run you would add `--shard i --num-shards N` to split the work over many
jobs, and `--push-to my-org/my-labels` to upload the result to the Hub.

In [ ]:
!python distill_dataset.py \
    --dataset {SRC_DATASET} --config {SRC_CONFIG} --split {SRC_SPLIT} \
    --language en --limit {N_TRAIN_CLIPS} --bs 2 --dtype {TEACHER_DTYPE} --no-streaming --out {RELABEL_DIR}

Now read the labelled dataset back the way `train.py` does, with `load_distill_dataset`. It drops rows that are too
short, too long, or have an empty teacher label, and it keeps the audio **encoded** (it is decoded per batch later), so
even a huge dataset costs almost no RAM.

Compare the teacher's text with the human reference. Notice the punctuation and casing: the teacher writes in its own
style, and the student will learn exactly that.

In [ ]:
from train import load_distill_dataset

dataset = load_distill_dataset(str(RELABEL_DIR), max_seconds=30.0)
assert len(dataset) >= BATCH_SIZE, "Too few usable training clips; increase N_TRAIN_CLIPS or reduce BATCH_SIZE."
print(dataset)
print(f"{len(dataset)} clips, {sum(dataset['duration']) / 60:.1f} minutes of audio\n")
for row in dataset.select(range(min(3, len(dataset)))):
    print("duration  :", round(row["duration"], 1), "s")
    print("reference :", row["reference"])
    print("teacher   :", row["teacher_text"])
    print("token ids :", row["teacher_ids"][:12], "...\n")

We also keep **held-out** clips that never go through training, for evaluation (step 3) and for the streaming demo
(step 4). We take the last `N_HELD_OUT` clips of the same small dataset; the relabelled ones came from the start of it.
We check that those index ranges do not overlap if you change the clip counts.
`Utterance` is the small container the evaluation code wants: an id, 16 kHz audio and the reference text.

In [ ]:
from datasets import Audio, load_dataset
from train import Utterance, audio_bytes, decode_flac

source = load_dataset(SRC_DATASET, SRC_CONFIG, split=SRC_SPLIT)
assert N_TRAIN_CLIPS > 0 and N_HELD_OUT > 0
assert N_TRAIN_CLIPS + N_HELD_OUT <= len(source), "Training and held-out ranges would overlap; reduce the clip counts."
source = source.cast_column("audio", Audio(decode=False))
held_out = source.select(range(len(source) - N_HELD_OUT, len(source)))

eval_utts = []
for index, row in enumerate(held_out):
    audio = decode_flac(audio_bytes(row["audio"]))
    eval_utts.append(Utterance(str(index), audio, row["text"]))

print(len(eval_utts), "held-out clips,", f"{sum(clip.seconds for clip in eval_utts):.0f} s in total")
print("first reference:", eval_utts[0].text)

## 2. Build the student and train it with `Trainer`

### 2a. The student

Teacher and student use compatible Conformer encoder blocks: a big **encoder** (audio to features) and a small
**decoder** (features to text, token by token). Cohere's weights initialize the streaming encoder, while its decoder
is retained. The encoder changes how it handles audio over time:

* each frame attends to a limited window of **past** frames (`left`, 70 frames = 5.6 s) plus a few **lookahead** frames,
* convolutions become causal and keep a small cache,
* the per-utterance feature normalisation becomes a **causal running** normalisation.

`train.build_student` does all of it: it loads the teacher, builds the streaming encoder **initialised from the
teacher's weights** (`model.build_streaming_encoder`) and wraps everything as a `StreamingStudent`. The command-line
flags of `train.py` live in the `DistillArguments` dataclass, so here we simply create one.

`StreamingStudent` trains with **three losses** added together (weights are the `*_weight` fields):

1. **AED loss** (`aed_weight=1.0`): cross-entropy of the decoder, reading the *student's* encoder features, against the
   teacher's tokens. The main objective: "say what the teacher would say".
2. **CTC loss** (`ctc_weight=0.3`): a new small output layer on the encoder learns to predict the same tokens. It
   gives fast first-pass partial results.
3. **KD loss** (`kd_weight=1.0`): encoder states at a few layers (`kd_layers`) must match the frozen teacher encoder's
   states on the same audio (*knowledge distillation*).

At every step a **random lookahead** from `lookaheads` is used, so one model works at all latency settings.

In [ ]:
from train import DistillArguments, build_student

if device != "cuda" or not NATIVE_BF16:
    raise RuntimeError("Student training needs a GPU with native bf16 support; select an A100 80 GB or H100 runtime.")

distill = DistillArguments(
    data=str(RELABEL_DIR),
    lookaheads=LOOKAHEADS,
    freeze_decoder=True,
    eval_lookaheads="0,6,13",
)
student, processor, encoder = build_student(distill, device)
student.encoder.gradient_checkpointing_enable()
student.teacher.to(torch.bfloat16)

def parameter_count_millions(parameters):
    return sum(parameter.numel() for parameter in parameters) / 1e6

trainable_parameters = (parameter for parameter in student.parameters() if parameter.requires_grad)
print(f"student encoder: {parameter_count_millions(student.encoder.parameters()):.0f} M parameters")
print(f"trainable      : {parameter_count_millions(trainable_parameters):.0f} M parameters")
print(f"frozen teacher : {parameter_count_millions(student.teacher.parameters()):.0f} M parameters")

### 2b. Data for training

Two more ingredients, both from the code of the real run:

* **Mel statistics.** The causal normaliser warms up from corpus-level mean/std of the log-mel features.
  `estimate_training_mel_stats` computes them from (up to 300) training clips. They are saved with the checkpoint,
  because streaming needs them at inference time.
* **`DistillCollator`**, the `data_collator` of the `Trainer`. It turns dataset rows into tensors: it decodes the audio,
  computes **two** feature versions of it (*streamable* causal-normalised features for the student, the teacher's
  native per-utterance-normalised features for the teacher), and builds the decoder inputs (`prompt + teacher tokens`)
  with shifted targets, plus the raw token ids for the CTC loss.

In [ ]:
from distill_dataset import DistillCollator
from train import estimate_training_mel_stats

stats = estimate_training_mel_stats(dataset, processor, seed=0)
print("mel statistics from", stats["frames"], "frames; mean[:4] =", [round(x, 2) for x in stats["mean"][:4]])

collator = DistillCollator(processor, stats, distill.norm, seed=0)
batch = collator([dataset[i] for i in range(BATCH_SIZE)])
for name, value in batch.items():
    print(f"{name:22s}", tuple(value.shape) if hasattr(value, "shape") else f"list of {len(value)} token lists")

Before training, run the student once on this batch at two lookaheads and look at the individual losses
(`StreamingStudent.forward` returns them in a dict). The KD loss is already small because the student starts from the
teacher's weights. The AED loss is higher, because the student must now work from causal features.

In [ ]:
inputs = {}
for name, value in batch.items():
    if isinstance(value, torch.Tensor):
        inputs[name] = value.to(device)
    else:
        inputs[name] = value

student.eval()
with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16, enabled=device == "cuda"):
    for lookahead in (0, 13):
        losses = student(**inputs, lookahead=lookahead)
        rounded_losses = {name: round(float(value), 3) for name, value in losses.items() if name != "lookahead"}
        print(f"lookahead {lookahead:<2}", rounded_losses)
del inputs, losses

### 2c. Baseline WER before training

We use `train.evaluate`, the same function the `Trainer` calls every few hundred steps in the real run (there it is
driven by `--eval_sets`, e.g. `librispeech:test.other`). For each lookahead it decodes the held-out clips two ways, with
the **AED decoder** (also used for the default live captions) and with the **CTC head**, and returns the WER in
%, computed with the Open ASR Leaderboard's text normaliser. The CTC head is brand new, so expect it to be bad.

In [ ]:
from train import evaluate

EVAL_LA = [0, 6, 13]
eval_sets = {"heldout": (eval_utts, "en")}

def run_eval():
    return evaluate(
        student,
        processor,
        stats,
        distill.norm,
        eval_sets,
        EVAL_LA,
        device,
        torch.float32,
        prior_frames=distill.eval_prior,
    )

before = run_eval()
print("before training:", json.dumps(before))

### 2d. Train with `transformers.Trainer`

Distillation is an ordinary supervised training loop with a custom loss, so we use the standard
[`Trainer`](https://huggingface.co/docs/transformers/main_classes/trainer) and get everything else for free:
multi-GPU, bf16, gradient accumulation, learning-rate schedule, logging, checkpoints, resume and pushing to the Hub.

`train.py` adds only three things on top:

* `DistillTrainingArguments`: `TrainingArguments` with defaults suited to this recipe (so **every** `Trainer` flag
  still works),
* `DistillTrainer`: a `Trainer` with the distillation loss in `compute_loss`, per-loss logging
  (`aed_loss`, `ctc_loss`, `kd_loss`) and checkpoints in the layout `model.CohereStreaming` loads,
* `build_optimizer`: the optimizer chosen by `--optim` (AdamW by default), with a higher learning rate (`--head_lr`) for the freshly initialised CTC head. `DistillTrainer` creates it automatically.

We keep this run short: a few steps, no periodic evaluation (we run it ourselves), no workers. Metrics are logged
to the local Trackio project `cohere-streaming`. A longer run would also turn on `eval_strategy="steps"` and `push_to_hub=True`.

In [ ]:
from train import DistillTrainer, DistillTrainingArguments

args = DistillTrainingArguments(
    output_dir=str(CKPT),
    max_steps=MAX_STEPS,
    bf16=NATIVE_BF16,
    fp16=False,
    per_device_train_batch_size=BATCH_SIZE,
    warmup_steps=min(5, MAX_STEPS // 4),
    logging_steps=1,
    eval_strategy="no",
    eval_on_start=False,
    save_strategy="no",
    dataloader_num_workers=0,
    report_to="trackio",
    project="cohere-streaming",
)
trainer = DistillTrainer(
    model=student,
    args=args,
    train_dataset=dataset,
    data_collator=collator,
    processing_class=processor,
    distill_args=distill,
    stats=stats,
)
training_started = time.time()
trainer.train()
print(f"trained {MAX_STEPS} steps in {time.time() - training_started:.0f} s")

`logging_steps=1` logged every step. The `Trainer` keeps its history in `trainer.state.log_history`; `DistillTrainer`
added the three loss terms to it. Let's plot them.

In [ ]:
import matplotlib.pyplot as plt

history = [entry for entry in trainer.state.log_history if "aed_loss" in entry]
steps = [entry["step"] for entry in history]
figure, axes = plt.subplots(figsize=(6, 3.4))
for name in ("aed", "ctc", "kd"):
    loss_values = [entry[f"{name}_loss"] for entry in history]
    axes.plot(steps, loss_values, label=name)
axes.set_xlabel("step")
axes.set_ylabel("loss")
axes.set_yscale("log")
axes.set_title("training losses")
axes.legend()
plt.tight_layout()
plt.show()

The curves can be noisy because each step samples a lookahead and the batch is tiny. The CTC head starts from random
weights, so its loss may change rapidly. Compare the measured losses rather than expecting a particular ordering.

Now save the checkpoint. `trainer.save_model()` writes three files needed by `CohereStreaming`, plus the training log:

* `student.safetensors`: student encoder, decoder, and CTC weights; the teacher encoder is reloaded from the gated base model for training,
* `config.json`: the encoder config (it contains the attention window) and the training arguments,
* `mel_stats.json`: the mel statistics from above,
* `log.json`: recorded training and evaluation metrics.

In [ ]:
trainer.save_model()
for checkpoint_file in sorted(CKPT.iterdir()):
    print(f"{checkpoint_file.name:24s} {checkpoint_file.stat().st_size / 1e6:8.1f} MB")

## 3. Evaluate after training

Run the same evaluation on the same held-out clips and compare WER before and after training. A few steps on a handful
of clips demonstrate the workflow; they do not guarantee better recognition. A useful run needs enough training audio
and a representative held-out evaluation set.

In [ ]:
after = run_eval()
print(f"{'':8s} {'AED WER %':>20s} {'CTC WER %':>20s}")
print(f"{'':8s} {'before':>9s} {'after':>10s} {'before':>10s} {'after':>9s}")
for lookahead in EVAL_LA:
    before_scores = before[f"R{lookahead}"]
    after_scores = after[f"R{lookahead}"]
    print(
        f"R={lookahead:<5d} {before_scores['aed_wer']:>9.1f} {after_scores['aed_wer']:>10.1f} "
        f"{before_scores['ctc_wer']:>10.1f} {after_scores['ctc_wer']:>9.1f}"
    )

## 4. Stream with the checkpoint

The checkpoint folder is all `model.CohereStreaming` needs: pass its path instead of a Hub id. It rebuilds the
streaming encoder, loads `student.safetensors` and uses `mel_stats.json` for the causal normalisation. (A checkpoint
pushed to the Hub works the same way, with the repo id.)

First free the GPU memory held by the training objects. Then use the usual three calls: `reset`, `step` once per 80 ms
chunk, `finish` at the end. `lookahead=6` gives an encoder audio wait of 560 ms;
actual caption timing also includes compute and decoding. Since this student has seen
only a few clips, expect rough text, but the machinery is the same as the released model's.

In [ ]:
del trainer, student, encoder, batch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

from model import CohereStreaming

streamer = CohereStreaming(str(CKPT), lookahead=6, device=device, dtype=torch.bfloat16)
clip = eval_utts[0]
streamer.reset(language="en")
chunk_samples = 16000 * streamer.hop_ms // 1000
previous_text = None
for start in range(0, len(clip.audio), chunk_samples):
    chunk = clip.audio[start:start + chunk_samples]
    if len(chunk) < chunk_samples:
        chunk = np.pad(chunk, (0, chunk_samples - len(chunk)))
    text = streamer.step(chunk)
    if text != previous_text:
        audio_time = (start + chunk_samples) / 16000
        print(f"[{audio_time:5.2f}s] {text}")
        previous_text = text
print("FINAL    :", streamer.finish())
print("REFERENCE:", clip.text)

## Next steps

To train for longer, use the same scripts from a terminal:

```bash
python distill_dataset.py \
    --dataset facebook/multilingual_librispeech --config french --split train --text-column transcript \
    --language fr --push-to my-org/distill-mls-fr

accelerate launch train.py \
    --data my-org/distill-mls-fr --max_steps 2000 --report_to trackio --project cohere-streaming --output_dir results/train
```

`python train.py --help` lists the distillation options and standard `TrainingArguments` flags.
`python distill_dataset.py --help` lists the relabelling options.

[Stream_Cohere_Transcribe.ipynb](https://github.com/Deep-unlearning/smol-audio/blob/master/cohere_transcribe_streaming/Stream_Cohere_Transcribe.ipynb)
shows the released checkpoint and compares lookahead settings.
